# **Convert Konooz Dev Files to the Model Input Format**

In [ ]:
from pathlib import Path


input_folder = Path(r"/content/model3_predictions")

output_folder = Path(r"/content/model3_predictions_update")

output_folder.mkdir(parents=True, exist_ok=True)


def convert_file(input_path: Path, output_path: Path) -> None:
    converted_lines = []

    with input_path.open("r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            stripped_line = line.strip()

            if not stripped_line:
                converted_lines.append("")
                continue

            columns = stripped_line.split()

            if not columns:
                converted_lines.append("")
                continue

            token = columns[0]

            converted_lines.append(f"{token} O")

    with output_path.open("w", encoding="utf-8", newline="\n") as file:
        file.write("\n".join(converted_lines))

        file.write("\n")


txt_files = sorted(input_folder.glob("*.txt"))

if not txt_files:
    raise FileNotFoundError(
        f"No TXT files were found in: {input_folder}"
    )

for input_path in txt_files:
    output_path = output_folder / input_path.name
    convert_file(input_path, output_path)

    print(f"Converted: {input_path.name}")

print()
print(f"Number of converted files: {len(txt_files)}")
print(f"Output folder: {output_folder}")

In [ ]:
!find /content/konooz-model-input

In [ ]:
!zip -r /content/konooz-model-input.zip /content/konooz-model-input

In [ ]:
from google.colab import files
files.download("/content/konooz-model-input.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Phase 1: Baseline Code:**

# **Train Nested NER:**

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
# Remove existing package and clone again from Github
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER/')

In [ ]:
# Import train function
from arabiner.bin.train import main as train

In [ ]:
# Setup the model arguments
args_dict = {
    # Model output path to save artifacts and model predictions
    "output_path": "/content/output/",

    # train/test/validation data paths
    "train_path": "/content/ArabicNER/data/train.txt",
    "test_path": "/content/ArabicNER/data/test.txt",
    "val_path": "/content/ArabicNER/data/val.txt",

    # seed for randomization
    "seed": 1,

    "batch_size": 8,

    # Nmber of workers for the dataloader
    "num_workers": 1,

    # GPU/device Ids to train model on
    # For two GPUs use [0, 1]
    # For three GPUs use [0, 1, 2], etc.
    "gpus": [0],

    # Overwrite data in output_path directory specified above
    "overwrite": True,

    # How often to print the logs in terms of number of steps
    "log_interval": 10,

    # Data configuration
    # Here we specify the dataset class and there are two options:
    #  arabiner.data.datasets.DefaultDataset: for flat NER
    #  arabiner.data.datasets.NestedTagsDataset: for nested NER
    #
    # kwargs: keyword arguments to the dataset class
    # This notebook used the NestedTagsDataset for nested NER
    "data_config": {
        "fn": "arabiner.data.datasets.NestedTagsDataset",
        "kwargs": {"max_seq_len": 512}
    },

    # Neural net configuration
    # There are two NNs:
    #   arabiner.nn.BertSeqTagger: flat NER tagger
    #   arabiner.nn.BertNestedTagger: nested NER tagger
    #
    # kwargs: keyword arguments to the NN
    # This notebook uses BertNestedTagger for nested NER tagging
    "network_config": {
        "fn": "arabiner.nn.BertNestedTagger",
        "kwargs": {"dropout": 0.1, "bert_model": "aubmindlab/bert-base-arabertv2"}
    },

    # Model trainer configuration
    #
    #  arabiner.trainers.BertTrainer: for flat NER training
    #  arabiner.trainers.BertNestedTrainer: for nested NER training
    #
    # kwargs: keyword arguments to arabiner.trainers.BertTrainer
    #         additional arguments you can pass includes
    #           - clip: for gradient clpping
    #           - patience: number of epochs for early termination
    # This notebook uses BertNestedTrainer for nested NER training
    "trainer_config": {
        "fn": "arabiner.trainers.BertNestedTrainer",
        "kwargs": {"max_epochs": 50}
    },

    # Optimizer configuration
    # Our experiments use torch.optim.AdamW, however, you are free to pass
    # any other optmizers such as torch.optim.Adam or torch.optim.SGD
    # lr: learning rate
    # kwargs: keyword arguments to torch.optim.AdamW or whatever optimizer you use
    #
    # Additional optimizers can be found here:
    # https://pytorch.org/docs/stable/optim.html
    "optimizer": {
        "fn": "torch.optim.AdamW",
        "kwargs": {"lr": 0.0001}
    },

    # Learning rate scheduler configuration
    # You can pass a learning scheduler such as torch.optim.lr_scheduler.StepLR
    # kwargs: keyword arguments to torch.optim.AdamW or whatever scheduler you use
    #
    # Additional schedulers can be found here:
    # https://pytorch.org/docs/stable/optim.html
    "lr_scheduler": {
        "fn": "torch.optim.lr_scheduler.ExponentialLR",
        "kwargs": {"gamma": 1}
    },

    # Loss function configuration
    # We use cross entropy loss
    # kwargs: keyword arguments to torch.nn.CrossEntropyLoss or whatever loss function you use
    "loss": {
        "fn": "torch.nn.CrossEntropyLoss",
        "kwargs": {}
    }
}

# Convert args dictionary to argparse namespace
args = argparse.Namespace()
args.__dict__ = args_dict

In [ ]:
# Start training the model
train(args)

In [ ]:
!find /content/output

In [ ]:
!zip -r /content/output.zip /content/output

In [ ]:
from google.colab import files
files.download("/content/output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**تحميل أفضل F1:**

In [ ]:
# هون لازم ازبط موضوع الاستدعاء كمان
trainer.load(
    "/content/output/checkpoints/checkpoint_best_f1.pt"
)

**تحميل أفضل Loss:**

In [ ]:
trainer.load(
    "/content/output/checkpoints/checkpoint_best_loss.pt"
)

**تحميل آخر مودل:**

In [ ]:
trainer.load(
    "/content/output/checkpoints/checkpoint_last.pt"
)

# **Evaluation Nested NER:**

In [ ]:
# Verify that you have the GPU recognized
!nvidia-smi

In [ ]:
# Install dependencies compatible with current Colab
!pip install -q transformers==4.41.2
!pip install -q seqeval==1.2.2

In [ ]:
%cd /content
!rm -rf ArabicNER

!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER/')

In [ ]:
# Import train function
from arabiner.bin.eval import main as eval

In [ ]:
# Setup the evaluation arguments
args_dict = {
    # Output path to save logs, metrics and predictions
    "output_path": "/content/outputEva/",

    # train/test/validation data paths
    # The data provided in the ArabicNER repo is a sample data
    # data_paths takes a list of data paths in case you need to evaluate multiple datasets
    "data_paths": ["/content/ArabicNER/data/test.txt"],

    # Path to the model, this corresponds to the "output_path" you specified
    # during training the model
    "model_path": "/content/output/",

    "batch_size": 8
}

# Convert args dictionary to argparse namespace
args = argparse.Namespace()
args.__dict__ = args_dict

In [ ]:
import sys
import types

torchtext = types.ModuleType("torchtext")
torchtext.__path__ = []

torchtext_vocab = types.ModuleType("torchtext.vocab")
torchtext_vocab.__path__ = []

torchtext_vocab_vocab = types.ModuleType("torchtext.vocab.vocab")
torchtext_internal = types.ModuleType("torchtext._torchtext")

class Vocab:
    def __init__(self, *args, **kwargs):
        self.stoi = {}
        self.itos = []

    def __len__(self):
        return len(self.itos)

    def get_stoi(self):
        return self.stoi

    def get_itos(self):
        return self.itos

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)

            # compatibility
            if not hasattr(self, "itos"):
                self.itos = []

            if not hasattr(self, "stoi"):
                self.stoi = {}

        elif isinstance(state, (list, tuple)):
            self.itos = state[0]
            self.stoi = {v: i for i, v in enumerate(self.itos)}

    def __getstate__(self):
        return self.__dict__



torchtext_vocab.Vocab = Vocab
torchtext_vocab_vocab.Vocab = Vocab
torchtext_internal.Vocab = Vocab

sys.modules["torchtext"] = torchtext
sys.modules["torchtext.vocab"] = torchtext_vocab
sys.modules["torchtext.vocab.vocab"] = torchtext_vocab_vocab
sys.modules["torchtext._torchtext"] = torchtext_internal

In [ ]:
import sys

for m in list(sys.modules.keys()):
    if m.startswith("torchtext"):
        del sys.modules[m]

In [ ]:
!pip uninstall -y torchtext
!pip install torchtext==0.6.0

In [ ]:
import importlib
import arabiner.utils.helpers

importlib.reload(arabiner.utils.helpers)

from arabiner.bin.eval import main as eval

In [ ]:
eval(args)

In [ ]:
!find /content/outputEva

In [ ]:
!zip -r /content/outputEva.zip /content/outputEva

In [ ]:
from google.colab import files
files.download("/content/outputEva.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Phase 2: Domain Adaptation to Unlabeled Konooz Texts**

# **1. Preparing Konooz Texts for Masked Language Modeling (MLM)**

In [ ]:
from pathlib import Path


KONOOZ_DIR = Path("/content/konooz")

OUTPUT_FILE = Path("/content/konooz_mlm_corpus.txt")


def read_konooz_file(file_path: Path) -> list[str]:
    """
    Reads one Konooz CoNLL file and reconstructs its sentences.

    Each non-empty line must contain:
        1 token + 21 placeholder tag columns.
    """
    sentences = []
    current_tokens = []

    with file_path.open("r", encoding="utf-8-sig") as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()

            # Empty line means end of sentence
            if not line:
                if current_tokens:
                    sentences.append(" ".join(current_tokens))
                    current_tokens = []
                continue

            parts = line.split()

            # Expected format: token + 21 tag columns
            if len(parts) != 22:
                raise ValueError(
                    f"Invalid format in {file_path.name}, "
                    f"line {line_number}: expected 22 values, "
                    f"but found {len(parts)}.\n"
                    f"Line content: {line}"
                )

            token = parts[0]
            current_tokens.append(token)

        # Save final sentence if file does not end with blank line
        if current_tokens:
            sentences.append(" ".join(current_tokens))

    return sentences


if not KONOOZ_DIR.exists():
    raise FileNotFoundError(
        f"Konooz directory was not found: {KONOOZ_DIR}"
    )

text_files = sorted(KONOOZ_DIR.glob("*.txt"))

if len(text_files) != 10:
    print(
        f"Warning: expected 10 domain files, "
        f"but found {len(text_files)}."
    )

all_sentences = []
domain_statistics = {}

for file_path in text_files:
    sentences = read_konooz_file(file_path)

    domain_statistics[file_path.stem] = {
        "sentences": len(sentences),
        "tokens": sum(len(sentence.split()) for sentence in sentences),
    }

    all_sentences.extend(sentences)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as output_file:
    for sentence in all_sentences:
        output_file.write(sentence + "\n")

print("=" * 60)
print(f"Processed domain files: {len(text_files)}")
print(f"Total sentences: {len(all_sentences)}")
print(f"Output file: {OUTPUT_FILE}")
print("=" * 60)

for domain, stats in domain_statistics.items():
    print(
        f"{domain:15s} | "
        f"Sentences: {stats['sentences']:4d} | "
        f"Tokens: {stats['tokens']:6d}"
    )

# **2. TAPT-adapted AraBERT**

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from pathlib import Path

MODEL_NAME = "aubmindlab/bert-base-arabertv2"

CORPUS_PATH = Path("/content/konooz_mlm_corpus.txt")
OUTPUT_DIR = Path("/content/arabert_konooz_tapt")

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Corpus file not found: {CORPUS_PATH}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Corpus:", CORPUS_PATH)
print("Output:", OUTPUT_DIR)

In [ ]:
from datasets import Dataset

with CORPUS_PATH.open("r", encoding="utf-8") as file:
    texts = [
        line.strip()
        for line in file
        if line.strip()
    ]

print("Total sentences:", len(texts))

dataset = Dataset.from_dict({
    "text": texts
})

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42,
    shuffle=True
)

print(dataset)
print("Train sentences:", len(dataset["train"]))
print("Validation sentences:", len(dataset["test"]))

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
)

print("Tokenizer vocabulary size:", len(tokenizer))
print("Model loaded successfully.")

In [ ]:
# Tokenization
MAX_LENGTH = 128

def tokenize_with_overflow(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_overflowing_tokens=True,
        return_special_tokens_mask=True
    )

tokenized_dataset = dataset.map(
    tokenize_with_overflow,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing and chunking Konooz"
)

print(tokenized_dataset)
print("Train chunks:", len(tokenized_dataset["train"]))
print("Validation chunks:", len(tokenized_dataset["test"]))

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["overflow_to_sample_mapping"]
)

print(tokenized_dataset)

In [ ]:
# Dynamic Masking
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
    return_tensors="pt"
)

In [ ]:
import shutil
import torch
from pathlib import Path
from transformers import TrainingArguments

OUTPUT_DIR = Path("/content/arabert_konooz_tapt")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=10,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=5e-6,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    seed=42,
    data_seed=42,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

print(training_args)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

train_result = trainer.train()

In [ ]:
eval_results = trainer.evaluate()

print("Final evaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

FINAL_MODEL_DIR = OUTPUT_DIR / "best_model"

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print("Adapted model saved to:")
print(FINAL_MODEL_DIR)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)
print("Final training epoch:", trainer.state.epoch)

In [ ]:
!find /content/arabert_konooz_tapt

In [ ]:
!zip -r /content/arabert_konooz_tapt.zip /content/arabert_konooz_tapt

In [ ]:
from google.colab import files
files.download("/content/arabert_konooz_tapt.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **3. Fine-Tuning the Nested NER Model on the Wojood Dataset**

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER-copy/')

In [ ]:
import os

print(os.path.exists("/content/ArabicNER-copy/arabert_konooz_tapt/best_model"))
print(os.listdir("/content/ArabicNER-copy/arabert_konooz_tapt/best_model"))

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
# Import train function
from arabiner.bin.train import main as train

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

In [ ]:
from pathlib import Path
from transformers import (
    AutoModelForMaskedLM,
    BertTokenizer
)

TAPT_MLM_PATH = Path(
    "/content/ArabicNER-copy/"
    "arabert_konooz_tapt/best_model"
)

TAPT_ENCODER_PATH = Path(
    "/content/ArabicNER-copy/"
    "arabert_konooz_tapt/encoder_model"
)

BASE_MODEL_NAME = "aubmindlab/bert-base-arabertv2"

TAPT_ENCODER_PATH.mkdir(
    parents=True,
    exist_ok=True
)

mlm_model = AutoModelForMaskedLM.from_pretrained(
    str(TAPT_MLM_PATH),
    local_files_only=True
)

encoder_model = mlm_model.bert

encoder_model.save_pretrained(
    str(TAPT_ENCODER_PATH),
    safe_serialization=True
)

tokenizer = BertTokenizer.from_pretrained(
    BASE_MODEL_NAME
)

tokenizer.save_pretrained(
    str(TAPT_ENCODER_PATH)
)

print("Encoder saved to:", TAPT_ENCODER_PATH)
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
import os

for file_name in sorted(os.listdir(TAPT_ENCODER_PATH)):
    print(file_name)

In [ ]:
from transformers import BertModel, BertTokenizer

test_tokenizer = BertTokenizer.from_pretrained(
    str(TAPT_ENCODER_PATH)
)

test_model = BertModel.from_pretrained(
    str(TAPT_ENCODER_PATH),
    add_pooling_layer=False
)

print("Encoder loaded successfully")
print("Vocabulary size:", test_tokenizer.vocab_size)

In [ ]:
import argparse

TAPT_MODEL_PATH = "/content/ArabicNER-copy/arabert_konooz_tapt/encoder_model"

# Setup the model arguments
args_dict = {
    "output_path": "/content/arabert_konooz_tapt/",

    # train/test/validation data paths
    "train_path": "/content/ArabicNER-copy/data/train.txt",
    "test_path": "/content/ArabicNER-copy/data/test.txt",
    "val_path": "/content/ArabicNER-copy/data/val.txt",

    # seed for randomization
    "seed": 1,

    "batch_size": 8,
    "num_workers": 1,
    "gpus": [0],

    # Overwrite data in output_path
    "overwrite": True,

    "log_interval": 10,

    # Data configuration
    "data_config": {
        "fn": "arabiner.data.datasets.NestedTagsDataset",
        "kwargs": {
            "max_seq_len": 512,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    # Neural network configuration
    "network_config": {
        "fn": "arabiner.nn.BertNestedTagger",
        "kwargs": {
            "dropout": 0.1,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    # Trainer configuration
    "trainer_config": {
        "fn": "arabiner.trainers.BertNestedTrainer",
        "kwargs": {
            "max_epochs": 50
        }
    },

    "optimizer": {
        "fn": "torch.optim.AdamW",
        "kwargs": {
            "lr": 1e-5
        }
    },

    "lr_scheduler": {
        "fn": "torch.optim.lr_scheduler.ExponentialLR",
        "kwargs": {
            "gamma": 1
        }
    },

    "loss": {
        "fn": "torch.nn.CrossEntropyLoss",
        "kwargs": {}
    }
}

# Convert args dictionary to argparse namespace
args = argparse.Namespace()
args.__dict__ = args_dict

In [ ]:
print("Model used:", args.network_config["kwargs"]["bert_model"])
print("Dataset model:", args.data_config["kwargs"]["bert_model"])
print("Output:", args.output_path)

In [ ]:
!find /content/ArabicNER-copy/arabert_konooz_tapt/encoder_model

In [ ]:
!zip -r /content/encoder_model.zip /content/ArabicNER-copy/arabert_konooz_tapt/encoder_model

In [ ]:
from google.colab import files
files.download("/content/encoder_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
train(args)

In [ ]:
!find /content/output_tapt_full_ner_new

In [ ]:
!zip -r /content/output_tapt_full_ner_new.zip /content/output_tapt_full_ner_new

In [ ]:
from google.colab import files
files.download("/content/output_tapt_full_ner_new.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Phase 3: R-Drop**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
OUTPUT_PATH = (
    "/content/drive/MyDrive/ArabicNER_outputs/"
    "output_tapt_rdrop_strong_augmentation_seed2026/"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER-copy/')

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
# Import train function
from arabiner.bin.train import main as train

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

In [ ]:
from arabiner.trainers import RDropBertNestedTrainer

print("RDropBertNestedTrainer imported successfully")

In [ ]:
import argparse
import os

TAPT_MODEL_PATH = (
    "/content/ArabicNER-copy/arabert_konooz_tapt/encoder_model"
)

if not os.path.isdir(TAPT_MODEL_PATH):
    raise FileNotFoundError(
        f"TAPT encoder was not found: {TAPT_MODEL_PATH}"
    )

args_dict_rdrop = {
    "output_path": OUTPUT_PATH,

    # "train_path": "/content/ArabicNER-copy/data/train.txt", # for model 1 + model 2
    "train_path": "/content/ArabicNER-copy/data/train_augmented.txt", # for model 3
    "test_path": "/content/ArabicNER-copy/data/test.txt",
    "val_path": "/content/ArabicNER-copy/data/val.txt",

    # "seed": 1,   # for model 1
    # "seed": 42,   # for model 2
    "seed": 2026,   # for model 3
    "batch_size": 8,
    "num_workers": 1,
    "gpus": [0],

    "overwrite": True,
    "log_interval": 10,

    "data_config": {
        "fn": "arabiner.data.datasets.NestedTagsDataset",
        "kwargs": {
            "max_seq_len": 512,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    "network_config": {
        "fn": "arabiner.nn.BertNestedTagger",
        "kwargs": {
            "dropout": 0.1,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    "trainer_config": {
        "fn": "arabiner.trainers.RDropBertNestedTrainer",
        "kwargs": {
            "max_epochs": 50,
            "patience": 5,
            "rdrop_alpha": 1.0
        }
    },

    "optimizer": {
        "fn": "torch.optim.AdamW",
        "kwargs": {
            "lr": 1e-5
        }
    },

    "lr_scheduler": {
        "fn": "torch.optim.lr_scheduler.ExponentialLR",
        "kwargs": {
            "gamma": 1
        }
    },

    "loss": {
        "fn": "torch.nn.CrossEntropyLoss",
        "kwargs": {}
    }
}

args_rdrop = argparse.Namespace()
args_rdrop.__dict__ = args_dict_rdrop

print("Model:", TAPT_MODEL_PATH)
print("Output:", args_rdrop.output_path)
print("Network:", args_rdrop.network_config["fn"])
print("Trainer:", args_rdrop.trainer_config["fn"])
print(
    "R-Drop alpha:",
    args_rdrop.trainer_config["kwargs"]["rdrop_alpha"]
)

In [ ]:
train(args_rdrop)

In [ ]:
import shutil

folder_path = "/content/drive/MyDrive/ArabicNER_outputs/output_tapt_rdrop_strong_augmentation_seed2026"
zip_path = "/content/drive/MyDrive/output_tapt_rdrop_strong_augmentation_seed2026"

shutil.make_archive(zip_path, 'zip', folder_path)

# **Phase 4: Stronger Data Augmentation (Tag-Preserving Word Dropout)**

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER/')

**1. Configuration**

In [ ]:
# Strong Data Augmentation for Nested Arabic NER

from pathlib import Path
from collections import Counter
import random


# File Paths
INPUT_FILE = Path(
    "/content/ArabicNER/data/train.txt"
)

OUTPUT_FILE = Path(
    "/content/ArabicNER/data/train_augmented.txt"
)


# General Configuration
RANDOM_SEED = 2026

AUGMENT_SENTENCE_RATIO = 0.30

DROPOUT_WEIGHT = 0.70
MASKING_WEIGHT = 0.15
SWAP_WEIGHT = 0.15


# Dropout Configuration
#

DROP_PROBABILITY = 0.10

MAX_REMOVAL_RATIO = 0.30


# Masking Configuration

MASK_PROBABILITY = 0.10

MAX_MASK_RATIO = 0.30

MASK_TOKEN = "[MASK]"


# Swap Configuration

MAX_SWAP_OPERATIONS = 1


rng = random.Random(RANDOM_SEED)


# Validate Configuration
operation_weight_sum = (
    DROPOUT_WEIGHT
    + MASKING_WEIGHT
    + SWAP_WEIGHT
)

assert abs(operation_weight_sum - 1.0) < 1e-9, (
    "Operation weights must sum to 1.0."
)

assert 0 <= AUGMENT_SENTENCE_RATIO <= 1
assert 0 <= DROP_PROBABILITY <= 1
assert 0 <= MASK_PROBABILITY <= 1
assert 0 <= MAX_REMOVAL_RATIO < 1
assert 0 <= MAX_MASK_RATIO <= 1


print("=" * 60)
print("Strong Augmentation Configuration")
print("=" * 60)

print("Input file              :", INPUT_FILE)
print("Output file             :", OUTPUT_FILE)
print("Random seed             :", RANDOM_SEED)
print("Sentence selection ratio:", AUGMENT_SENTENCE_RATIO)

print()
print("Dropout weight          :", DROPOUT_WEIGHT)
print("Masking weight          :", MASKING_WEIGHT)
print("Swap weight             :", SWAP_WEIGHT)

print()
print("Drop probability        :", DROP_PROBABILITY)
print("Mask probability        :", MASK_PROBABILITY)
print("Mask token              :", MASK_TOKEN)

**2. Read the Wojood Training File**

In [ ]:
# Read CoNLL-style Nested NER File

def read_conll_file(file_path):
    """
    Read a Nested NER file.

    Expected format:
        token tag1 tag2 tag3 ...

    Blank lines separate sentences.

    Returns:
        List of sentences.

    Each sentence:
        List of rows.

    Each row:
        [token, tag1, tag2, ...]
    """

    sentences = []
    current_sentence = []

    with open(
        file_path,
        mode="r",
        encoding="utf-8-sig"
    ) as file:

        for line_number, raw_line in enumerate(
            file,
            start=1
        ):

            line = raw_line.strip()

            # Blank line separates sentences
            if not line:

                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []

                continue

            columns = line.split()

            if len(columns) < 2:
                raise ValueError(
                    f"Invalid row at line {line_number}.\n"
                    f"Expected token plus at least one tag.\n"
                    f"Found: {raw_line!r}"
                )

            current_sentence.append(columns)

    # Add last sentence if file has no final blank line
    if current_sentence:
        sentences.append(current_sentence)

    return sentences


# Load Original Training Data

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Training file was not found:\n{INPUT_FILE}"
    )


original_sentences = read_conll_file(
    INPUT_FILE
)


number_of_original_tokens = sum(
    len(sentence)
    for sentence in original_sentences
)


print("=" * 60)
print("Original Training Dataset")
print("=" * 60)

print("Sentences:", len(original_sentences))
print("Tokens   :", number_of_original_tokens)

**3. Helper Functions**

In [ ]:
# Helper Functions

def copy_sentence(sentence):
    """
    Create a deep-enough copy of a sentence.

    Each row is copied independently.
    """

    return [
        row.copy()
        for row in sentence
    ]


def is_all_o(token_columns):
    """
        ex:

        ["في", "O", "O"] -> True

        ["محمد", "B-PERS", "O"] -> False

        ["فلسطين", "O", "B-GPE"] -> False
    """

    labels = token_columns[1:]

    return (
        len(labels) > 0
        and all(label == "O" for label in labels)
    )


def get_all_o_indices(sentence):
    """
    Return indices of tokens labeled O
    in all Nested NER columns.
    """

    return [
        index
        for index, row in enumerate(sentence)
        if is_all_o(row)
    ]


# Punctuation Filtering for Safer Swap
PUNCTUATION_TOKENS = {
    ".", "،", ",", "؛", ";", ":", "!", "؟", "?",
    "(", ")", "[", "]", "{", "}",
    "\"", "'", "«", "»", "-", "–", "—",
    "/", "\\", "...", "…", "ـ", "``", "''",
    "\"\"\"\"", "“", "”", "‘", "’"
}


def is_punctuation_token(token_text):
    """
    Return True if the token is punctuation or consists
    only of non-alphanumeric symbols.
    """

    token_text = token_text.strip()

    if not token_text:
        return True

    if token_text in PUNCTUATION_TOKENS:
        return True

    return not any(
        character.isalnum()
        for character in token_text
    )


def is_valid_swap_token(token_columns):
    
    token_text = token_columns[0]

    return (
        is_all_o(token_columns)
        and not is_punctuation_token(token_text)
        and any(
            character.isalpha()
            for character in token_text
        )
    )


def get_adjacent_all_o_pairs(sentence):
   

    adjacent_pairs = []

    for index in range(len(sentence) - 1):

        first_row = sentence[index]
        second_row = sentence[index + 1]

        if (
            is_valid_swap_token(first_row)
            and is_valid_swap_token(second_row)
        ):
            adjacent_pairs.append(
                (index, index + 1)
            )

    return adjacent_pairs


def get_entity_rows(sentence):
    
    return [
        row.copy()
        for row in sentence
        if not is_all_o(row)
    ]


def choose_augmentation_operation():
    """
        Dropout: 70%
        Masking: 15%
        Swap:    15%
    """

    return rng.choices(
        population=[
            "dropout",
            "masking",
            "swap"
        ],
        weights=[
            DROPOUT_WEIGHT,
            MASKING_WEIGHT,
            SWAP_WEIGHT
        ],
        k=1
    )[0]

**4. O-only Word Dropout**

In [ ]:
# O-only Word Dropout

def apply_o_only_dropout(sentence):
    

    original_copy = copy_sentence(sentence)

    sentence_length = len(sentence)

    if sentence_length <= 1:
        return original_copy, None

    eligible_indices = get_all_o_indices(
        sentence
    )

    if not eligible_indices:
        return original_copy, None

    max_removed = int(
        sentence_length * MAX_REMOVAL_RATIO
    )

    max_removed = min(
        max_removed,
        len(eligible_indices),
        sentence_length - 1
    )

    if max_removed < 1:
        return original_copy, None

    selected_indices = [
        index
        for index in eligible_indices
        if rng.random() < DROP_PROBABILITY
    ]

    # Ensure that the augmented copy is different
    if not selected_indices:
        selected_indices = [
            rng.choice(eligible_indices)
        ]

    # Enforce maximum removal ratio
    if len(selected_indices) > max_removed:

        selected_indices = rng.sample(
            selected_indices,
            max_removed
        )

    selected_indices = sorted(
        set(selected_indices)
    )

    selected_indices_set = set(
        selected_indices
    )

    removed_rows = [
        sentence[index].copy()
        for index in selected_indices
    ]

    augmented_sentence = [
        row.copy()
        for index, row in enumerate(sentence)
        if index not in selected_indices_set
    ]

    if not augmented_sentence:
        return original_copy, None

    details = {
        "operation": "dropout",
        "changed_indices": selected_indices,
        "removed_rows": removed_rows,
        "change_count": len(selected_indices)
    }

    return augmented_sentence, details

**5. O-only Word Masking**

In [ ]:
# O-only Word Masking

def apply_o_only_masking(sentence):
    
    augmented_sentence = copy_sentence(
        sentence
    )

    sentence_length = len(sentence)

    if sentence_length == 0:
        return augmented_sentence, None

    eligible_indices = [
        index
        for index, row in enumerate(sentence)
        if (
            is_all_o(row)
            and any(
                character.isalpha()
                for character in row[0]
            )
        )
    ]

    if not eligible_indices:
        return augmented_sentence, None

    max_masked = int(
        sentence_length * MAX_MASK_RATIO
    )

    max_masked = min(
        max_masked,
        len(eligible_indices)
    )

    if max_masked < 1:
        return augmented_sentence, None

    selected_indices = [
        index
        for index in eligible_indices
        if rng.random() < MASK_PROBABILITY
    ]

    # Ensure a real change
    if not selected_indices:

        selected_indices = [
            rng.choice(eligible_indices)
        ]

    # Enforce maximum masking ratio
    if len(selected_indices) > max_masked:

        selected_indices = rng.sample(
            selected_indices,
            max_masked
        )

    selected_indices = sorted(
        set(selected_indices)
    )

    original_rows = []

    for index in selected_indices:

        original_rows.append(
            sentence[index].copy()
        )

        # Change token text only
        augmented_sentence[index][0] = MASK_TOKEN

        # Labels remain exactly unchanged
        assert (
            augmented_sentence[index][1:]
            == sentence[index][1:]
        )

    details = {
        "operation": "masking",
        "changed_indices": selected_indices,
        "original_rows": original_rows,
        "mask_token": MASK_TOKEN,
        "change_count": len(selected_indices)
    }

    return augmented_sentence, details

**6. O-only Adjacent Word Swap**

In [ ]:
# O-only Adjacent Word Swap

def apply_o_only_swap(sentence):
     

    augmented_sentence = copy_sentence(
        sentence
    )

    adjacent_pairs = get_adjacent_all_o_pairs(
        sentence
    )

    if not adjacent_pairs:
        return augmented_sentence, None

    first_index, second_index = rng.choice(
        adjacent_pairs
    )

    first_original_row = sentence[
        first_index
    ].copy()

    second_original_row = sentence[
        second_index
    ].copy()

    # Swap complete rows
    augmented_sentence[first_index], \
    augmented_sentence[second_index] = (
        augmented_sentence[second_index],
        augmented_sentence[first_index]
    )

    # If both rows are identical, the result did not change
    if augmented_sentence == sentence:
        return copy_sentence(sentence), None

    details = {
        "operation": "swap",
        "changed_indices": [
            first_index,
            second_index
        ],
        "swap_pair": (
            first_index,
            second_index
        ),
        "original_rows": [
            first_original_row,
            second_original_row
        ],
        "change_count": 1
    }

    return augmented_sentence, details

**7. Unified Strong Augmentation Function**

In [ ]:
# Apply One Random Strong Augmentation Operation

def apply_strong_augmentation(
    sentence,
    operation
):
     

    if operation == "dropout":

        return apply_o_only_dropout(
            sentence
        )

    if operation == "masking":

        return apply_o_only_masking(
            sentence
        )

    if operation == "swap":

        return apply_o_only_swap(
            sentence
        )

    raise ValueError(
        f"Unsupported augmentation operation: "
        f"{operation}"
    )

**8. Create the Augmented Training Dataset**

In [ ]:
# Create Strongly Augmented Training Dataset

final_sentences = []

# Additional augmented copies only
augmented_sentences = []

# Records used later for detailed validation
augmentation_records = []


selected_sentence_count = 0
successful_augmentation_count = 0
skipped_augmentation_count = 0


selected_operation_counter = Counter()
successful_operation_counter = Counter()


for sentence_index, sentence in enumerate(
    original_sentences
):

    # Always preserve the original sentence
    original_copy = copy_sentence(
        sentence
    )

    final_sentences.append(
        original_copy
    )

    # Select approximately 30% of sentences
    if rng.random() >= AUGMENT_SENTENCE_RATIO:
        continue

    selected_sentence_count += 1

    # Select one operation using 70/15/15 weights
    operation = choose_augmentation_operation()

    selected_operation_counter[
        operation
    ] += 1

    # Apply the selected operation
    augmented_sentence, details = (
        apply_strong_augmentation(
            sentence=sentence,
            operation=operation
        )
    )

    # The selected operation may not be possible:
    # e.g. swap when there are no adjacent all-O tokens.
    if (
        details is None
        or augmented_sentence == sentence
    ):

        skipped_augmentation_count += 1
        continue

    # Store the successful augmented copy
    augmented_copy = copy_sentence(
        augmented_sentence
    )

    final_sentences.append(
        augmented_copy
    )

    augmented_sentences.append(
        augmented_copy
    )

    augmentation_records.append({
        "sentence_index": sentence_index,
        "operation": operation,
        "original": original_copy,
        "augmented": augmented_copy,
        "details": details
    })

    successful_augmentation_count += 1

    successful_operation_counter[
        operation
    ] += 1


# Summary
actual_augmentation_ratio = (
    successful_augmentation_count
    / len(original_sentences)
    if original_sentences
    else 0
)


print("=" * 65)
print("Strong Augmentation Summary")
print("=" * 65)

print(
    f"Original sentences          : "
    f"{len(original_sentences)}"
)

print(
    f"Selected sentences          : "
    f"{selected_sentence_count}"
)

print(
    f"Successful augmented copies : "
    f"{successful_augmentation_count}"
)

print(
    f"Skipped selected sentences  : "
    f"{skipped_augmentation_count}"
)

print(
    f"Final training sentences    : "
    f"{len(final_sentences)}"
)

print(
    f"Actual augmentation ratio   : "
    f"{actual_augmentation_ratio:.2%}"
)


print()
print("Selected operation counts:")

for operation in [
    "dropout",
    "masking",
    "swap"
]:
    print(
        f"  {operation:10}: "
        f"{selected_operation_counter[operation]}"
    )


print()
print("Successful operation counts:")

for operation in [
    "dropout",
    "masking",
    "swap"
]:
    print(
        f"  {operation:10}: "
        f"{successful_operation_counter[operation]}"
    )

**9. Save the Augmented File**

In [ ]:
# Save Strongly Augmented Training File

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


with open(
    OUTPUT_FILE,
    mode="w",
    encoding="utf-8",
    newline="\n"
) as file:

    for sentence in final_sentences:

        for token_columns in sentence:

            file.write(
                " ".join(token_columns)
                + "\n"
            )

        # Blank line between sentences
        file.write("\n")


print("=" * 60)
print("Augmented File Saved")
print("=" * 60)

print(OUTPUT_FILE)

**10. Dataset Statistics**

In [ ]:
# Dataset Statistics

def dataset_statistics(sentences):
    """
    Calculate general Nested NER dataset statistics.
    """

    stats = {
        "sentences": len(sentences),
        "tokens": 0,
        "all_o_tokens": 0,
        "entity_tokens": 0,
        "entity_tags": 0,
        "mask_tokens": 0
    }

    entity_tag_counter = Counter()

    for sentence in sentences:

        stats["tokens"] += len(sentence)

        for row in sentence:

            token_text = row[0]
            labels = row[1:]

            if is_all_o(row):
                stats["all_o_tokens"] += 1
            else:
                stats["entity_tokens"] += 1

            if token_text == MASK_TOKEN:
                stats["mask_tokens"] += 1

            for label in labels:

                if label != "O":

                    stats["entity_tags"] += 1
                    entity_tag_counter[label] += 1

    return stats, entity_tag_counter


original_stats, original_tag_counter = (
    dataset_statistics(
        original_sentences
    )
)

augmented_stats, augmented_tag_counter = (
    dataset_statistics(
        augmented_sentences
    )
)

final_stats, final_tag_counter = (
    dataset_statistics(
        final_sentences
    )
)


def print_statistics(title, stats):

    print("=" * 60)
    print(title)
    print("=" * 60)

    for key, value in stats.items():
        print(f"{key:20}: {value}")

    print()


print_statistics(
    "Original Dataset",
    original_stats
)

print_statistics(
    "Augmented Copies Only",
    augmented_stats
)

print_statistics(
    "Final Training Dataset",
    final_stats
)

**11. Validate Dropout Records**

In [ ]:
# Dropout Validation

def validate_dropout_record(record):
    """
    Validate one dropout augmentation record.
    """

    original = record["original"]
    augmented = record["augmented"]
    details = record["details"]

    removed_indices = details[
        "changed_indices"
    ]

    removed_indices_set = set(
        removed_indices
    )

    # There must be at least one removed token
    if not removed_indices:
        return False, "No token was removed."

    # Every removed token must be all-O
    for index in removed_indices:

        if not is_all_o(original[index]):
            return (
                False,
                f"Entity token removed at index {index}."
            )

    # Reconstruct expected sentence
    expected_augmented = [
        row.copy()
        for index, row in enumerate(original)
        if index not in removed_indices_set
    ]

    if expected_augmented != augmented:
        return (
            False,
            "Augmented sentence does not match "
            "the recorded removed indices."
        )

    # Sentence must not be empty
    if not augmented:
        return False, "Empty sentence generated."

    # Check maximum removal ratio
    removal_ratio = (
        len(removed_indices)
        / len(original)
    )

    if removal_ratio > (
        MAX_REMOVAL_RATIO + 1e-12
    ):
        return (
            False,
            f"Removal ratio exceeded: "
            f"{removal_ratio:.2%}"
        )

    # Entity rows and their order must remain unchanged
    if (
        get_entity_rows(original)
        != get_entity_rows(augmented)
    ):
        return (
            False,
            "Entity rows changed during dropout."
        )

    return True, "Valid"

**12. Validate Masking Records**

In [ ]:
# Masking Validation

def validate_masking_record(record):
    """
    Validate one masking augmentation record.
    """

    original = record["original"]
    augmented = record["augmented"]
    details = record["details"]

    changed_indices = details[
        "changed_indices"
    ]

    changed_indices_set = set(
        changed_indices
    )

    if len(original) != len(augmented):
        return (
            False,
            "Masking changed the sentence length."
        )

    if not changed_indices:
        return (
            False,
            "No token was masked."
        )

    for index in range(len(original)):

        original_row = original[index]
        augmented_row = augmented[index]

        if index in changed_indices_set:

            # Original token must be all-O
            if not is_all_o(original_row):
                return (
                    False,
                    f"Entity token masked at index {index}."
                )

            # Token text must become [MASK]
            if augmented_row[0] != MASK_TOKEN:
                return (
                    False,
                    f"Incorrect mask token at index {index}."
                )

            # Tags must remain unchanged
            if (
                original_row[1:]
                != augmented_row[1:]
            ):
                return (
                    False,
                    f"Tags changed at index {index}."
                )

        else:

            # Unselected rows must remain identical
            if original_row != augmented_row:
                return (
                    False,
                    f"Unexpected modification at index "
                    f"{index}."
                )

    mask_ratio = (
        len(changed_indices)
        / len(original)
    )

    if mask_ratio > (
        MAX_MASK_RATIO + 1e-12
    ):
        return (
            False,
            f"Mask ratio exceeded: "
            f"{mask_ratio:.2%}"
        )

    if (
        get_entity_rows(original)
        != get_entity_rows(augmented)
    ):
        return (
            False,
            "Entity rows changed during masking."
        )

    return True, "Valid"

**13. Validate Swap Records**

In [ ]:
# Swap Validation

def validate_swap_record(record):
    """
    Validate one adjacent all-O swap record.
    """

    original = record["original"]
    augmented = record["augmented"]
    details = record["details"]

    first_index, second_index = details[
        "swap_pair"
    ]

    # Sentence length must not change
    if len(original) != len(augmented):
        return (
            False,
            "Swap changed the sentence length."
        )

    # Tokens must be adjacent
    if second_index != first_index + 1:
        return (
            False,
            "Swapped tokens were not adjacent."
        )

    if not is_valid_swap_token(original[first_index]):
        return (
            False,
            "First swapped token is not a valid "
            "all-O word token."
        )

    if not is_valid_swap_token(original[second_index]):
        return (
            False,
            "Second swapped token is not a valid "
            "all-O word token."
        )

    # Create expected result
    expected_augmented = copy_sentence(
        original
    )

    expected_augmented[first_index], \
    expected_augmented[second_index] = (
        expected_augmented[second_index],
        expected_augmented[first_index]
    )

    if expected_augmented != augmented:
        return (
            False,
            "Swap result does not match the "
            "recorded adjacent pair."
        )

    # All non-swapped rows must remain unchanged
    for index in range(len(original)):

        if index not in {
            first_index,
            second_index
        }:

            if original[index] != augmented[index]:
                return (
                    False,
                    f"Unexpected row modification at "
                    f"index {index}."
                )

    # Entity order and rows must remain unchanged
    if (
        get_entity_rows(original)
        != get_entity_rows(augmented)
    ):
        return (
            False,
            "Entity rows changed during swap."
        )

    return True, "Valid"

**14. Complete Sanity Check**

In [ ]:
# Complete Strong Augmentation Sanity Check

validation_errors = []


for record_number, record in enumerate(
    augmentation_records
):

    operation = record["operation"]

    if operation == "dropout":

        is_valid, message = (
            validate_dropout_record(record)
        )

    elif operation == "masking":

        is_valid, message = (
            validate_masking_record(record)
        )

    elif operation == "swap":

        is_valid, message = (
            validate_swap_record(record)
        )

    else:

        is_valid = False
        message = (
            f"Unknown operation: {operation}"
        )

    if not is_valid:

        validation_errors.append({
            "record_number": record_number,
            "sentence_index": record[
                "sentence_index"
            ],
            "operation": operation,
            "message": message
        })


# Global Dataset Assertions
expected_final_sentence_count = (
    len(original_sentences)
    + len(augmented_sentences)
)

assert len(final_sentences) == (
    expected_final_sentence_count
), (
    "Final sentence count is incorrect."
)


assert final_stats["tokens"] == (
    original_stats["tokens"]
    + augmented_stats["tokens"]
), (
    "Final token count is incorrect."
)


assert final_stats["entity_tokens"] == (
    original_stats["entity_tokens"]
    + augmented_stats["entity_tokens"]
), (
    "Final entity-token count is incorrect."
)


assert final_stats["entity_tags"] == (
    original_stats["entity_tags"]
    + augmented_stats["entity_tags"]
), (
    "Final entity-tag count is incorrect."
)


assert len(augmentation_records) == (
    successful_augmentation_count
), (
    "Augmentation record count is incorrect."
)


assert sum(
    successful_operation_counter.values()
) == successful_augmentation_count, (
    "Successful operation counts are incorrect."
)


assert not validation_errors, (
    "Strong augmentation validation failed.\n"
    f"First errors:\n{validation_errors[:5]}"
)


# ==========================================================
# Verification Report
# ==========================================================

print("=" * 65)
print("Strong Augmentation Sanity Check")
print("=" * 65)

print("All original sentences were preserved.")
print("Only additional augmented copies were added.")
print("Every augmented copy contains a real change.")
print("No entity token was deleted.")
print("No entity token was masked.")
print("No entity token was moved.")
print("No B- or I- tag was changed.")
print("All Dropout removals were all-O tokens.")
print("All masked tokens were originally all-O.")
print("All Swap pairs were adjacent all-O tokens.")
print("Entity rows and their order were preserved.")
print("Maximum Dropout and Masking limits were respected.")
print("Dataset sentence and token counts are consistent.")

print()
print(
    "Valid augmented copies:",
    len(augmentation_records)
)

**15. Reload and Verify the Saved File**

In [ ]:
# Reload and Verify Saved File
saved_sentences = read_conll_file(
    OUTPUT_FILE
)


assert len(saved_sentences) == len(
    final_sentences
), (
    "Saved sentence count does not match "
    "the generated dataset."
)


assert saved_sentences == final_sentences, (
    "Saved file content differs from "
    "the generated data."
)


saved_stats, saved_tag_counter = (
    dataset_statistics(
        saved_sentences
    )
)


assert saved_stats == final_stats, (
    "Saved file statistics do not match."
)


assert saved_tag_counter == final_tag_counter, (
    "Saved file entity-tag distribution "
    "does not match."
)


print("=" * 60)
print("Saved File Validation")
print("=" * 60)

print("Saved file was read successfully.")
print("Saved sentences match generated sentences.")
print("Tokens and Nested BIO tags were preserved.")
print("Entity-tag distribution is correct.")

print()
print("Final augmented file:")
print(OUTPUT_FILE)

**16. Preview Examples from Each Operation**

In [ ]:
# Preview Examples from Each Operation

EXAMPLES_PER_OPERATION = 3


for operation_name in [
    "dropout",
    "masking",
    "swap"
]:

    operation_records = [
        record
        for record in augmentation_records
        if record["operation"] == operation_name
    ]

    print()
    print("=" * 70)
    print(
        f"{operation_name.upper()} Examples"
    )
    print("=" * 70)

    if not operation_records:

        print(
            f"No successful {operation_name} "
            f"examples were generated."
        )

        continue

    for example_number, record in enumerate(
        operation_records[
            :EXAMPLES_PER_OPERATION
        ],
        start=1
    ):

        original_tokens = [
            row[0]
            for row in record["original"]
        ]

        augmented_tokens = [
            row[0]
            for row in record["augmented"]
        ]

        print()
        print("-" * 70)
        print(f"Example {example_number}")
        print("-" * 70)

        print(
            "Sentence index:",
            record["sentence_index"]
        )

        print(
            "Operation     :",
            record["operation"]
        )

        print()
        print(
            "Original :",
            " ".join(original_tokens)
        )

        print()
        print(
            "Augmented:",
            " ".join(augmented_tokens)
        )

        if operation_name == "dropout":

            removed_tokens = [
                row[0]
                for row in record[
                    "details"
                ]["removed_rows"]
            ]

            print()
            print(
                "Removed:",
                " | ".join(removed_tokens)
            )

        elif operation_name == "masking":

            print()
            print(
                "Masked indices:",
                record[
                    "details"
                ]["changed_indices"]
            )

        elif operation_name == "swap":

            print()
            print(
                "Swapped indices:",
                record[
                    "details"
                ]["swap_pair"]
            )

**17. Final Operation Distribution**

In [ ]:
# Final Operation Distribution

print("=" * 65)
print("Successful Augmentation Distribution")
print("=" * 65)

total_successful = sum(
    successful_operation_counter.values()
)


for operation in [
    "dropout",
    "masking",
    "swap"
]:

    count = successful_operation_counter[
        operation
    ]

    percentage = (
        count / total_successful * 100
        if total_successful
        else 0
    )

    print(
        f"{operation:10}: "
        f"{count:6} copies "
        f"({percentage:6.2f}%)"
    )


print()
print("Target distribution:")
print("dropout   : 70%")
print("masking   : 15%")
print("swap      : 15%")

# **Phase 6: Label Konooz Using the Trained NER Model**

# **Automatic Labeling of Unlabeled Konooz Data:**

In [ ]:
!pip install -q transformers==4.41.2
!pip uninstall -y torchtext
!pip install -q torchtext==0.6.0
!pip install -q seqeval==1.2.2

In [ ]:
%cd /content
!rm -rf ArabicNER-copy
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Configure the ArabicNER Package

import sys

repo_path = "/content/ArabicNER-copy"

if repo_path not in sys.path:
    sys.path.append(repo_path)

print("ArabicNER repository added to system path.")

In [ ]:
# Import Required Libraries

import os
import argparse
import importlib
from pathlib import Path

import arabiner.utils.helpers

importlib.reload(arabiner.utils.helpers)

from arabiner.bin.predict import main as predict_model

In [ ]:
# Set the Final TAPT and R-Drop Model Path

model_path = "/content/ArabicNER-copy/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed2026"
print("Model path:", model_path)
print("Model files:")

!find "$model_path" -maxdepth 2 -type f

In [ ]:
# Collect All Unlabeled Konooz Files

konooz_folder = Path(
    "/content/ArabicNER-copy/konooz-model-input"
)

data_paths = sorted(
    str(file_path)
    for file_path in konooz_folder.glob("*.txt")
)

print(f"Number of Konooz files: {len(data_paths)}")

for file_path in data_paths:
    print(file_path)

assert len(data_paths) > 0, "No Konooz text files were found."

In [ ]:
# Configure Automatic Labeling Arguments

output_path = "/content/Konooz_Pseudo_Labels_Knowledge_Distillationn_seed2026/"

args_dict = {
    "output_path": output_path,
    "data_paths": data_paths,
    "model_path": model_path,
    "batch_size": 8
}

args = argparse.Namespace(**args_dict)

print(args)

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

In [ ]:
predict_model(args)

In [ ]:
# Display Generated Prediction Files

prediction_files = sorted(
    Path(output_path).glob("*.txt")
)

print(f"Number of generated prediction files: {len(prediction_files)}")

for prediction_file in prediction_files:
    print(prediction_file.name)

assert len(prediction_files) == len(data_paths), (
    f"Expected {len(data_paths)} prediction files, "
    f"but found {len(prediction_files)}."
)

In [ ]:
# Preview a Generated Prediction File

if prediction_files:
    sample_prediction = prediction_files[0]

    print("File:", sample_prediction.name)
    print("-" * 80)

    with open(sample_prediction, "r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            print(line, end="")

            if index >= 30:
                break

In [ ]:
# Compress the Generated Prediction Files

!rm -f /content/Konooz_Pseudo_Labels_tapt_rdrop.zip

!zip -r \
    /content/Konooz_Pseudo_Labels_tapt_rdrop.zip \
    /content/Konooz_Pseudo_Labels_2

In [ ]:
# Download the Generated Prediction Files

from google.colab import files

files.download(
    "/content/Konooz_Pseudo_Labels_tapt_rdrop.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Convert and Merge Konooz Predictions for Final Submission**

In [ ]:
from pathlib import Path
import zipfile

In [ ]:
# Folder containing the 10 prediction files
predictions_folder = Path("/content/predictions")
submission_file = Path(r"/content/yafa_pred_RDROP_seed2026.txt")

# ZIP file to upload
zip_file = Path(r"/content/AseelQararia_submission2026.zip")

In [ ]:
# Official domain order (DO NOT CHANGE)

prediction_files = [
    "Agriculture.txt",
    "Art.txt",
    "Economics.txt",
    "Finance.txt",
    "Health.txt",
    "History.txt",
    "Law.txt",
    "Politics.txt",
    "Science.txt",
    "Sport.txt",
]


EXPECTED_TAG_COLUMNS = 21
EXPECTED_TOTAL_COLUMNS = 22


def convert_prediction_file(input_path: Path):
    converted_lines = []

    previous_blank = False

    with input_path.open("r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            stripped = line.strip()

            # Keep only one blank line
            if not stripped:
                if converted_lines and not previous_blank:
                    converted_lines.append("")
                previous_blank = True
                continue

            previous_blank = False

            # Skip header
            if stripped.startswith("Token\tGold Tag\tPredicted Tag"):
                continue

            columns = stripped.split("\t", maxsplit=2)

            if len(columns) != 3:
                raise ValueError(
                    f"{input_path.name} "
                    f"(line {line_number}) "
                    f"does not contain exactly 3 columns."
                )

            token = columns[0].strip()

            predicted_tags = [
                tag.strip()
                for tag in columns[2].split("|")
            ]

            if len(predicted_tags) != EXPECTED_TAG_COLUMNS:
                raise ValueError(
                    f"{input_path.name} "
                    f"(line {line_number}) "
                    f"contains {len(predicted_tags)} tags "
                    f"instead of {EXPECTED_TAG_COLUMNS}."
                )

            output = [token] + predicted_tags

            if len(output) != EXPECTED_TOTAL_COLUMNS:
                raise ValueError(
                    f"Token '{token}' "
                    f"does not contain exactly "
                    f"{EXPECTED_TOTAL_COLUMNS} columns."
                )

            converted_lines.append(" ".join(output))

    while converted_lines and converted_lines[-1] == "":
        converted_lines.pop()

    return converted_lines


In [ ]:
# Merge all domains

all_lines = []

for file_name in prediction_files:

    input_path = predictions_folder / file_name

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    domain_lines = convert_prediction_file(input_path)

    if all_lines:
        all_lines.append("")

    all_lines.extend(domain_lines)

    print(f"Processed: {file_name}")


submission_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

with submission_file.open(
    "w",
    encoding="utf-8",
    newline="\n"
) as file:

    file.write("\n".join(all_lines))
    file.write("\n")

print()
print(f"Submission file created:\n{submission_file}")

In [ ]:
# Create ZIP
with zipfile.ZipFile(
    zip_file,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_archive:

    zip_archive.write(
        submission_file,
        arcname=submission_file.name
    )

print(f"ZIP file created:\n{zip_file}")

In [ ]:
from google.colab import files
files.download("/content/AseelQararia_submission2026.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Phase 7: Weighted Logits Ensemble**

model 1: TAPT+ R-Drop with seed = 1

model 2: TAPT+ R-Drop with seed = 42

model 3: TAPT+ R-Drop + Augmentation with seed = 2026

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER-copy/')

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

In [ ]:
import os

ENSEMBLE_FILE = (
    "/content/ArabicNER-copy/"
    "arabiner/bin/predict_ensemble.py"
)

print("File exists:", os.path.isfile(ENSEMBLE_FILE))
print("Path:", ENSEMBLE_FILE)

In [ ]:
from arabiner.bin import predict_ensemble

print("predict_ensemble imported successfully")

In [ ]:
import glob
import os

DATA_DIR = "/content/ArabicNER-copy/konooz-model-input"

data_paths = sorted(
    glob.glob(os.path.join(DATA_DIR, "*.txt"))
)

print("Number of files:", len(data_paths))

for path in data_paths:
    print(os.path.basename(path))

In [ ]:
%cd /content/ArabicNER-copy

!rm -rf /content/ensemble_check_model3_after_update

!python -m arabiner.bin.predict_ensemble \
  --output_path "/content/ensemble_check_model3_after_update" \
  --model_paths \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop" \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop_seed42" \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop_strong_augmentation_seed2026" \
  --model_weights 0.10 0.20 0.70 
  --ensemble_type logits \
  --data_paths /content/ArabicNER-copy/konooz-model-input/*.txt \
  --batch_size 8

In [ ]:
from pathlib import Path

probability_folder = Path(
    "/content/ensemble_check_model3_after_update"
)

probability_files = sorted(
    probability_folder.glob("predictions_*.txt")
)

print("Number of files:", len(probability_files))

for path in probability_files:
    print(path.name)

assert len(probability_files) == 10

print(
    "Weighted probability predictions "
    "were generated successfully."
)

In [ ]:
# Compress the Generated Prediction Files

!rm -f /content/Konooz_Pseudo_Labels_ensemble.zip

!zip -r \
    /content/Konooz_Pseudo_Labels_ensemble.zip \
    /content/ensemble_check_model3_after_update

In [ ]:
# Download the Generated Prediction Files

from google.colab import files

files.download(
    "/content/Konooz_Pseudo_Labels_ensemble.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Final System Development and Submission Pipeline**

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER-copy/')

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

# **1. Soft Teacher Output Generation**

In [ ]:
import os

ENSEMBLE_FILE = (
    "/content/ArabicNER-copy/"
    "arabiner/bin/predict_ensemble.py"
)

print("File exists:", os.path.isfile(ENSEMBLE_FILE))
print("Path:", ENSEMBLE_FILE)

In [ ]:
import glob
import os

DATA_DIR = "/content/ArabicNER-copy/konooz-model-input"

data_paths = sorted(
    glob.glob(os.path.join(DATA_DIR, "*.txt"))
)

print("Number of files:", len(data_paths))

for path in data_paths:
    print(os.path.basename(path))

In [ ]:
%cd /content/ArabicNER-copy

!rm -rf /content/ensemble_teacher_outputs

!python -m arabiner.bin.predict_ensemble \
  --output_path "/content/ensemble_teacher_outputs" \
  --model_paths \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop" \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop_seed42" \
    "/content/ArabicNER-copy/Phase1_Output/output_tapt_rdrop_strong_augmentation_seed2026" \
  --model_weights 0.10 0.20 0.70 \
  --ensemble_type logits \
  --data_paths /content/ArabicNER-copy/konooz-model-input/*.txt \
  --batch_size 8 \
  --save_teacher_outputs

In [ ]:
import os
import glob

files = sorted(
    glob.glob(
        "/content/ensemble_teacher_outputs/"
        "teacher_outputs_*.pt"
    )
)

print("Number of teacher files:", len(files))

for file_path in files:
    print(
        os.path.basename(file_path),
        os.path.getsize(file_path) / (1024 ** 2),
        "MB",
    )

In [ ]:
import torch

teacher_data = torch.load(
    files[0],
    map_location="cpu",
    weights_only=False,
)

print("Keys:")
print(teacher_data.keys())

print("\nNumber of sentences:")
print(len(teacher_data["probabilities"]))

print("\nFirst sentence shapes:")
print(
    "logits:",
    teacher_data["logits"][0].shape,
)
print(
    "probabilities:",
    teacher_data["probabilities"][0].shape,
)
print(
    "confidence:",
    teacher_data["confidence"][0].shape,
)
print(
    "entropy:",
    teacher_data["entropy"][0].shape,
)
print(
    "predictions:",
    teacher_data["predictions"][0].shape,
)

In [ ]:
# Compress the Generated Prediction Files

!rm -f /content/Konooz_Pseudo_Labels_ensemble.zip

!zip -r \
    /content/Konooz_Pseudo_Labels_ensemble.zip \
    /content/ensemble_teacher_outputs

In [ ]:
# Download the Generated Prediction Files

from google.colab import files

files.download(
    "/content/Konooz_Pseudo_Labels_ensemble.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **2. Span-Level Confidence Analysis and Filtering**

In [ ]:
%cd /content/ArabicNER-copy
!rm -rf /content/span_confidence_analysis

!python -m arabiner.bin.analyze_teacher_spans \
  --teacher_dir "/content/ArabicNER-copy/ensemble_teacher_outputs" \
  --output_dir "/content/span_confidence_analysis"

In [ ]:
report_path = (
    "/content/span_confidence_analysis/"
    "span_threshold_report.txt"
)

with open(
    report_path,
    "r",
    encoding="utf-8",
) as file:
    report = file.read()

print(report)

# **3. Build the Filtered Teacher Dataset**

In [ ]:
%cd /content/ArabicNER-copy

!rm -rf /content/filtered_teacher_dataset

!python -m arabiner.bin.build_filtered_teacher_dataset \
  --teacher_dir "/content/ArabicNER-copy/ensemble_teacher_outputs" \
  --output_dir "/content/filtered_teacher_dataset" \
  --o_confidence_threshold 0.97 \
  --o_entropy_threshold 0.20 \
  --o_to_entity_ratio 2.0 \
  --seed 2026

In [ ]:
report_path = (
    "/content/filtered_teacher_dataset/"
    "filtered_teacher_report.txt"
)

with open(
    report_path,
    "r",
    encoding="utf-8",
) as file:
    report = file.read()

print(report)

In [ ]:
import torch
import glob

filtered_files = sorted(
    glob.glob(
        "/content/filtered_teacher_dataset/"
        "filtered_teacher_*.pt"
    )
)

filtered_files = [
    path
    for path in filtered_files
    if not path.endswith(
        "filtered_teacher_manifest.pt"
    )
]

print(
    "Number of domain files:",
    len(filtered_files),
)

data = torch.load(
    filtered_files[0],
    map_location="cpu",
    weights_only=False,
)

print("Keys:")
print(data.keys())

print("\nDomain:")
print(data["domain"])

print("\nNumber of sentences:")
print(len(data["sentences"]))

first_sentence = data["sentences"][0]

print("\nFirst sentence keys:")
print(first_sentence.keys())

print("\nFirst sentence shapes:")
for key in [
    "probabilities",
    "predictions",
    "confidence",
    "entropy",
    "entity_mask",
    "o_mask",
    "distillation_mask",
]:
    print(
        key,
        first_sentence[key].shape,
    )

print(
    "\nAccepted spans in first sentence:",
    len(first_sentence["accepted_spans"]),
)

print(
    "Entity positions:",
    int(
        first_sentence[
            "entity_mask"
        ].sum().item()
    ),
)

print(
    "O positions:",
    int(
        first_sentence[
            "o_mask"
        ].sum().item()
    ),
)

print(
    "Total distillation positions:",
    int(
        first_sentence[
            "distillation_mask"
        ].sum().item()
    ),
)

In [ ]:
# Compress the Generated Prediction Files

!rm -f /content/filtered_teacher_dataset.zip

!zip -r \
    /content/filtered_teacher_dataset.zip \
    /content/filtered_teacher_dataset

In [ ]:
# Download the Generated Prediction Files

from google.colab import files

files.download(
    "/content/filtered_teacher_dataset.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **5. Train the Student Models with R-Drop and Knowledge Distillation**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
OUTPUT_PATH = (
    "/content/drive/MyDrive/ArabicNER_outputs/"
    "output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed2026/"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
%cd /content
!rm -rf ArabicNER
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/aseeljaber18-jpg/ArabicNER-copy.git

In [ ]:
# Add the ArabicNER package to the system path
import sys
import argparse
sys.path.append('/content/ArabicNER-copy/')

In [ ]:
# Install dependencies compatible with current Colab

!pip install -q transformers==4.40.2
!pip install -q seqeval==1.2.2

In [ ]:
# Import train function
from arabiner.bin.train import main as train

In [ ]:
!apt-get update -qq
!apt-get install -y git-lfs

!cd /content/ArabicNER-copy && git lfs install
!cd /content/ArabicNER-copy && git lfs pull

In [ ]:
import os

model_file = "/content/ArabicNER-copy/arabert_konooz_tapt/best_model/model.safetensors"

print("Size (MB):", os.path.getsize(model_file) / (1024 ** 2))

In [ ]:
from arabiner.trainers import RDropBertNestedTrainer

print("RDropBertNestedTrainer imported successfully")

In [ ]:
import argparse
import os

TAPT_MODEL_PATH = (
    "/content/ArabicNER-copy/arabert_konooz_tapt/encoder_model"
)

if not os.path.isdir(TAPT_MODEL_PATH):
    raise FileNotFoundError(
        f"TAPT encoder was not found: {TAPT_MODEL_PATH}"
    )

args_dict_rdrop = {
    "output_path": OUTPUT_PATH,

    # "train_path": "/content/ArabicNER-copy/data/train.txt", # for model 1 + model 2
    "train_path": "/content/ArabicNER-copy/data/train_augmented.txt", # for model 3
    "test_path": "/content/ArabicNER-copy/data/test.txt",
    "val_path": "/content/ArabicNER-copy/data/val.txt",
    "teacher_data_paths": [
        "/content/ArabicNER-copy/konooz-model-input/Agriculture.txt",
        "/content/ArabicNER-copy/konooz-model-input/Art.txt",
        "/content/ArabicNER-copy/konooz-model-input/Economics.txt",
        "/content/ArabicNER-copy/konooz-model-input/Finance.txt",
        "/content/ArabicNER-copy/konooz-model-input/Health.txt",
        "/content/ArabicNER-copy/konooz-model-input/History.txt",
        "/content/ArabicNER-copy/konooz-model-input/Law.txt",
        "/content/ArabicNER-copy/konooz-model-input/Politics.txt",
        "/content/ArabicNER-copy/konooz-model-input/Science.txt",
        "/content/ArabicNER-copy/konooz-model-input/Sport.txt",
    ],

    "filtered_teacher_dir": (
        "/content/ArabicNER-copy/"
        "filtered_teacher_dataset"
    ),

    "teacher_batch_size": 8,

    # "seed": 1,   # for model 1
    # "seed": 42,   # for model 2
    "seed": 2026,   # for model 3
    "batch_size": 8,
    "num_workers": 1,
    "gpus": [0],

    "overwrite": True,
    "log_interval": 10,

    "data_config": {
        "fn": "arabiner.data.datasets.NestedTagsDataset",
        "kwargs": {
            "max_seq_len": 512,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    "network_config": {
        "fn": "arabiner.nn.BertNestedTagger",
        "kwargs": {
            "dropout": 0.1,
            "bert_model": TAPT_MODEL_PATH
        }
    },

    "trainer_config": {
        "fn": "arabiner.trainers.RDropBertNestedTrainer",
        "kwargs": {
            "max_epochs": 50,
            "patience": 5,
            "rdrop_alpha": 1.0,
            "distillation_alpha": 0.5,
            "distillation_temperature": 2.0
        }
    },

    "optimizer": {
        "fn": "torch.optim.AdamW",
        "kwargs": {
            "lr": 1e-5
        }
    },

    "lr_scheduler": {
        "fn": "torch.optim.lr_scheduler.ExponentialLR",
        "kwargs": {
            "gamma": 1
        }
    },

    "loss": {
        "fn": "torch.nn.CrossEntropyLoss",
        "kwargs": {}
    }
}

args_rdrop = argparse.Namespace()
args_rdrop.__dict__ = args_dict_rdrop

print("Model:", TAPT_MODEL_PATH)
print("Output:", args_rdrop.output_path)
print("Network:", args_rdrop.network_config["fn"])
print("Trainer:", args_rdrop.trainer_config["fn"])
print(
    "R-Drop alpha:",
    args_rdrop.trainer_config["kwargs"]["rdrop_alpha"]
)

In [ ]:
train(args_rdrop)

In [ ]:
import shutil

folder_path = "/content/drive/MyDrive/ArabicNER_outputs/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed2026"
zip_path = "/content/drive/MyDrive/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed2026"

shutil.make_archive(zip_path, 'zip', folder_path)

# **6. Entity-Specific Ensemble Calibration**

# **6.1 Per-Entity Validation Evaluation**

In [ ]:
# model 1
import argparse

args_eval_model1 = argparse.Namespace(
    output_path=(
        "/content/ArabicNER-copy/"
        "per_entity_eval_model1"
    ),
    model_path=(
        "/content/ArabicNER-copy/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed1"
    ),
    data_paths=[
        "/content/ArabicNER-copy/data/val.txt"
    ],
    batch_size=8,
)

from arabiner.bin.eval import main as eval_model

eval_model(args_eval_model1)

In [ ]:
# model 2
import argparse

args_eval_model2 = argparse.Namespace(
    output_path=(
        "/content/ArabicNER-copy/"
        "per_entity_eval_model2"
    ),
    model_path=(
        "/content/ArabicNER-copy/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed42"
    ),
    data_paths=[
        "/content/ArabicNER-copy/data/val.txt"
    ],
    batch_size=8,
)

from arabiner.bin.eval import main as eval_model

eval_model(args_eval_model2)

In [ ]:
# model 3
import argparse

args_eval_model3 = argparse.Namespace(
    output_path=(
        "/content/ArabicNER-copy/"
        "per_entity_eval_model3"
    ),
    model_path=(
        "/content/ArabicNER-copy/output_tapt_rdrop_strong_augmentation_Knowledge_Distillationn_seed2026"
    ),
    data_paths=[
        "/content/ArabicNER-copy/data/val.txt"
    ],
    batch_size=8,
)

from arabiner.bin.eval import main as eval_model

eval_model(args_eval_model3)

# **6.2 Compute Support-Aware Entity-Specific Calibration Weights**

In [ ]:
import json
import math
import os


METRICS_PATHS = {
    "model_1": (
        "/content/ArabicNER-copy/"
        "per_entity_eval_model1/metrics_val.json"
    ),
    "model_2": (
        "/content/ArabicNER-copy/"
        "per_entity_eval_model2/metrics_val.json"
    ),
    "model_3": (
        "/content/ArabicNER-copy/"
        "per_entity_eval_model3/metrics_val.json"
    ),
}


OUTPUT_PATH = (
    "/content/ArabicNER-copy/"
    "entity_specific_calibrated_weights.json"
)


MODEL_PATHS = {
    "model_1": (
        "/content/ArabicNER-copy/"
        "output_tapt_rdrop_strong_augmentation_"
        "Knowledge_Distillationn_seed1/"
        "checkpoints/checkpoint_best_f1.pt"
    ),
    "model_2": (
        "/content/ArabicNER-copy/"
        "output_tapt_rdrop_strong_augmentation_"
        "Knowledge_Distillationn_seed42/"
        "checkpoints/checkpoint_best_f1.pt"
    ),
    "model_3": (
        "/content/ArabicNER-copy/"
        "output_tapt_rdrop_strong_augmentation_"
        "Knowledge_Distillationn_seed2026/"
        "checkpoints/checkpoint_best_f1.pt"
    ),
}

CODABENCH_MICRO_F1 = {
    "model_1": 0.7098,
    "model_2": 0.6974,
    "model_3": 0.7069,
}

GLOBAL_PRIORS = {
    "model_1": 0.45,
    "model_2": 0.10,
    "model_3": 0.45,
}



ENTITY_SUPPORT = {
    "CARDINAL": 170,
    "CURR": 24,
    "DATE": 1691,
    "EVENT": 292,
    "FAC": 118,
    "GPE": 2156,
    "LANGUAGE": 16,
    "LAW": 47,
    "LOC": 90,
    "MONEY": 22,
    "NORP": 551,
    "OCC": 522,
    "ORDINAL": 544,
    "ORG": 1882,
    "PERCENT": 12,
    "PERS": 679,
    "PRODUCT": 8,
    "QUANTITY": 3,
    "TIME": 33,
    "UNIT": 4,
    "WEBSITE": 80,
}


TEMPERATURE = 0.03 

SUPPORT_SCALE = 100.0

MAX_MODEL_WEIGHT = 0.75


# 1. Validate metric files
for model_name, path in METRICS_PATHS.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{model_name} metrics file was not found:\n"
            f"{path}"
        )


# 2. Load metric files
metrics_data = {}

for model_name, path in METRICS_PATHS.items():
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as fh:
        metrics_data[model_name] = json.load(fh)

    if "per_entity" not in metrics_data[model_name]:
        raise KeyError(
            f"'per_entity' was not found in:\n"
            f"{path}"
        )


# 3. Check entity compatibility
reference_entities = list(
    metrics_data["model_1"]["per_entity"].keys()
)

reference_entity_set = set(reference_entities)

for model_name in (
    "model_2",
    "model_3",
):
    current_entities = set(
        metrics_data[
            model_name
        ]["per_entity"].keys()
    )

    if current_entities != reference_entity_set:
        missing_entities = (
            reference_entity_set
            - current_entities
        )

        extra_entities = (
            current_entities
            - reference_entity_set
        )

        raise ValueError(
            f"Entity mismatch for {model_name}.\n"
            f"Missing: {sorted(missing_entities)}\n"
            f"Extra: {sorted(extra_entities)}"
        )


missing_support = (
    reference_entity_set
    - set(ENTITY_SUPPORT.keys())
)

if missing_support:
    raise ValueError(
        "Support values are missing for:\n"
        f"{sorted(missing_support)}"
    )


# 4. Calculate weights
entity_weights = {}

for entity_type in reference_entities:

    entity_f1 = {
        model_name: float(
            metrics_data[
                model_name
            ]["per_entity"][
                entity_type
            ]["f1"]
        )
        for model_name in METRICS_PATHS
    }

    best_f1 = max(
        entity_f1.values()
    )

    # Raw per-entity softmax weights
    unnormalized_weights = {}

    for model_name, current_f1 in (
        entity_f1.items()
    ):
        relative_score = math.exp(
            (
                current_f1
                - best_f1
            )
            / TEMPERATURE
        )

        unnormalized_weights[
            model_name
        ] = (
            relative_score
            * GLOBAL_PRIORS[model_name]
        )

    raw_weight_sum = sum(
        unnormalized_weights.values()
    )

    raw_weights = {
        model_name: (
            unnormalized_weights[
                model_name
            ]
            / raw_weight_sum
        )
        for model_name in METRICS_PATHS
    }

    # Support-aware smoothing
    support = ENTITY_SUPPORT[
        entity_type
    ]

    reliability = (
        support
        / (
            support
            + SUPPORT_SCALE
        )
    )

    smoothed_weights = {}

    for model_name in METRICS_PATHS:
        smoothed_weights[
            model_name
        ] = (
            reliability
            * raw_weights[model_name]
            + (
                1.0
                - reliability
            )
            * GLOBAL_PRIORS[
                model_name
            ]
        )

    # Cap weights to avoid extreme dominance
    capped_weights = {
        model_name: min(
            smoothed_weights[
                model_name
            ],
            MAX_MODEL_WEIGHT,
        )
        for model_name in METRICS_PATHS
    }

    capped_sum = sum(
        capped_weights.values()
    )

    final_weights = {
        model_name: (
            capped_weights[
                model_name
            ]
            / capped_sum
        )
        for model_name in METRICS_PATHS
    }

    entity_weights[
        entity_type
    ] = {
        "model_1": final_weights[
            "model_1"
        ],
        "model_2": final_weights[
            "model_2"
        ],
        "model_3": final_weights[
            "model_3"
        ],
        "support": support,
        "reliability": reliability,
        "validation_f1": {
            "model_1": entity_f1[
                "model_1"
            ],
            "model_2": entity_f1[
                "model_2"
            ],
            "model_3": entity_f1[
                "model_3"
            ],
        },
        "raw_weights": {
            "model_1": raw_weights[
                "model_1"
            ],
            "model_2": raw_weights[
                "model_2"
            ],
            "model_3": raw_weights[
                "model_3"
            ],
        },
    }


# 5. Build output data
output_data = {
    "model_order": [
        "model_1",
        "model_2",
        "model_3",
    ],
    "model_paths": [
        MODEL_PATHS["model_1"],
        MODEL_PATHS["model_2"],
        MODEL_PATHS["model_3"],
    ],
    "calibration": {
        "method": (
            "support_aware_softmax_"
            "per_entity_f1_with_global_prior"
        ),
        "temperature": TEMPERATURE,
        "support_scale": SUPPORT_SCALE,
        "max_model_weight": (
            MAX_MODEL_WEIGHT
        ),
        "codabench_micro_f1": (
            CODABENCH_MICRO_F1
        ),
        "global_priors": (
            GLOBAL_PRIORS
        ),
    },
    "entity_weights": (
        entity_weights
    ),
}


# 6. Save JSON
with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        output_data,
        fh,
        ensure_ascii=False,
        indent=4,
    )


print(
    "Entity-specific calibrated "
    "weights created successfully."
)

print(
    "Output:",
    OUTPUT_PATH,
)

print(
    "Number of entity types:",
    len(entity_weights),
)


# 7. Display final weights

print("\n" + "=" * 85)

print(
    "FINAL SUPPORT-AWARE "
    "ENTITY-SPECIFIC WEIGHTS"
)

print("=" * 85)

for entity_type, values in (
    entity_weights.items()
):
    print(
        f"{entity_type:10s} | "
        f"M1={values['model_1']:.4f} | "
        f"M2={values['model_2']:.4f} | "
        f"M3={values['model_3']:.4f} | "
        f"support={values['support']:4d} | "
        f"reliability="
        f"{values['reliability']:.4f}"
    )


# 8. Validate weight sums
print("\n" + "=" * 85)

print("WEIGHT SUM VALIDATION")

print("=" * 85)

for entity_type, values in (
    entity_weights.items()
):
    current_sum = (
        values["model_1"]
        + values["model_2"]
        + values["model_3"]
    )

    status = (
        "Valid"
        if abs(
            current_sum - 1.0
        ) < 1e-6
        else "Invalid"
    )

    print(
        f"{status} "
        f"{entity_type:10s} "
        f"sum={current_sum:.8f}"
    )

# **6.3 Validate the Computed Calibration Weights**

In [ ]:
import json
import os

FINAL_WEIGHTS_PATH = (
    "/content/ArabicNER-copy/"
    "entity_specific_calibrated_weights.json"
)

if not os.path.isfile(FINAL_WEIGHTS_PATH):
    raise FileNotFoundError(
        "Final calibration weights file was not found:\n"
        f"{FINAL_WEIGHTS_PATH}"
    )

with open(
    FINAL_WEIGHTS_PATH,
    "r",
    encoding="utf-8",
) as fh:
    calibrated_data = json.load(fh)

print(
    "Calibration temperature:",
    calibrated_data["calibration"]["temperature"],
)

print(
    "Number of entity types:",
    len(calibrated_data["entity_weights"]),
)

print("\n" + "=" * 85)
print("FINAL ENTITY-SPECIFIC CALIBRATION WEIGHTS")
print("=" * 85)

for entity_type, values in (
    calibrated_data["entity_weights"].items()
):
    current_sum = (
        values["model_1"]
        + values["model_2"]
        + values["model_3"]
    )

    status = (
        "Valid"
        if abs(current_sum - 1.0) < 1e-6
        else "Invalid"
    )

    print(
        f"{status:7s} | "
        f"{entity_type:10s} | "
        f"M1={values['model_1']:.4f} | "
        f"M2={values['model_2']:.4f} | "
        f"M3={values['model_3']:.4f} | "
        f"sum={current_sum:.8f}"
    )

# **6.4 Generate Final Predictions Using the Confidence-Aware Entity-Specific Ensemble**

In [ ]:
import argparse
import glob

FINAL_WEIGHTS_PATH = (
    "/content/ArabicNER-copy/"
    "entity_specific_calibrated_weights.json"
)


args_final_entity_ensemble = argparse.Namespace(
    output_path=(
        "/content/ArabicNER-copy/"
        "final_entity_specific_confidence_ensemble"
    ),

    model_paths=[
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed1"
        ),
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed42"
        ),
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed2026"
        ),
    ],

    model_weights=[
        1.0,
        1.0,
        1.0,
    ],

    data_paths=sorted(
        glob.glob(
            "/content/ArabicNER-copy/"
            "konooz-model-input/*.txt"
        )
    ),

    batch_size=8,

    ensemble_type="logits",

    confidence_weighting=True,

    entity_weights_path=(
        FINAL_WEIGHTS_PATH
    ),

    save_teacher_outputs=False,
)


from arabiner.bin.predict_ensemble import (
    main as predict_final_entity_ensemble
)


predict_final_entity_ensemble(
    args_final_entity_ensemble
)

In [ ]:
import argparse

FINAL_WEIGHTS_PATH = (
    "/content/ArabicNER-copy/"
    "entity_specific_calibrated_weights.json"
)


args_final_entity_ensemble = argparse.Namespace(
    output_path=(
        "/content/ArabicNER-copy/"
        "final_entity_specific_confidence_ensemble"
    ),

    model_paths=[
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed1"
        ),
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed42"
        ),
        (
            "/content/ArabicNER-copy/"
            "output_tapt_rdrop_strong_augmentation_"
            "Knowledge_Distillationn_seed2026"
        ),
    ],

    model_weights=[
        1.0,
        1.0,
        1.0,
    ],

    data_paths=[
        "/content/ArabicNER-copy/data/train.txt"
    ],

    batch_size=8,

    ensemble_type="logits",

    confidence_weighting=True,

    entity_weights_path=(
        FINAL_WEIGHTS_PATH
    ),

    save_teacher_outputs=False,
)


from arabiner.bin.predict_ensemble import (
    main as predict_final_entity_ensemble
)


predict_final_entity_ensemble(
    args_final_entity_ensemble
)

# **6.5 Archive the Final Prediction Files**

In [ ]:
from google.colab import files
import shutil


folder_path = (
    "/content/ArabicNER-copy/"
    "final_entity_specific_confidence_ensemble"
)

zip_path = (
    "/content/"
    "final_entity_specific_confidence_ensemble.zip"
)


shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    folder_path,
)


print(
    "Final prediction archive created:",
    zip_path,
)


files.download(zip_path)

# **Convert and Merge Konooz Predictions for Final Submission**

In [ ]:
from pathlib import Path
import zipfile

In [ ]:
# Folder containing the 10 prediction files
predictions_folder = Path("/content/predictions3")
submission_file = Path(r"/content/yafa_pred_ensemble.txt")

# ZIP file to upload
zip_file = Path(r"/content/submission.zip")

In [ ]:
# Official domain order (DO NOT CHANGE)

prediction_files = [
    "Agriculture.txt",
    "Art.txt",
    "Economics.txt",
    "Finance.txt",
    "Health.txt",
    "History.txt",
    "Law.txt",
    "Politics.txt",
    "Science.txt",
    "Sport.txt",
]


EXPECTED_TAG_COLUMNS = 21
EXPECTED_TOTAL_COLUMNS = 22


def convert_prediction_file(input_path: Path):
    converted_lines = []

    previous_blank = False

    with input_path.open("r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            stripped = line.strip()

            # Keep only one blank line
            if not stripped:
                if converted_lines and not previous_blank:
                    converted_lines.append("")
                previous_blank = True
                continue

            previous_blank = False

            # Skip header
            if stripped.startswith("Token\tGold Tag\tPredicted Tag"):
                continue

            columns = stripped.split("\t", maxsplit=2)

            if len(columns) != 3:
                raise ValueError(
                    f"{input_path.name} "
                    f"(line {line_number}) "
                    f"does not contain exactly 3 columns."
                )

            token = columns[0].strip()

            predicted_tags = [
                tag.strip()
                for tag in columns[2].split("|")
            ]

            if len(predicted_tags) != EXPECTED_TAG_COLUMNS:
                raise ValueError(
                    f"{input_path.name} "
                    f"(line {line_number}) "
                    f"contains {len(predicted_tags)} tags "
                    f"instead of {EXPECTED_TAG_COLUMNS}."
                )

            output = [token] + predicted_tags

            if len(output) != EXPECTED_TOTAL_COLUMNS:
                raise ValueError(
                    f"Token '{token}' "
                    f"does not contain exactly "
                    f"{EXPECTED_TOTAL_COLUMNS} columns."
                )

            converted_lines.append(" ".join(output))

    while converted_lines and converted_lines[-1] == "":
        converted_lines.pop()

    return converted_lines


In [ ]:
# Merge all domains

all_lines = []

for file_name in prediction_files:

    input_path = predictions_folder / file_name

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    domain_lines = convert_prediction_file(input_path)

    if all_lines:
        all_lines.append("")

    all_lines.extend(domain_lines)

    print(f"Processed: {file_name}")


submission_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

with submission_file.open(
    "w",
    encoding="utf-8",
    newline="\n"
) as file:

    file.write("\n".join(all_lines))
    file.write("\n")

print()
print(f"Submission file created:\n{submission_file}")

In [ ]:
# Create ZIP
with zipfile.ZipFile(
    zip_file,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_archive:

    zip_archive.write(
        submission_file,
        arcname=submission_file.name
    )

print(f"ZIP file created:\n{zip_file}")

In [ ]:
from google.colab import files
files.download("/content/submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>